In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("misrakahmed/vegetable-image-dataset")

print("Path to dataset files:", path)

100%|██████████| 534M/534M [00:06<00:00, 83.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/misrakahmed/vegetable-image-dataset/versions/1


In [2]:
!cp -r /root/.cache/kagglehub/datasets/misrakahmed/vegetable-image-dataset/versions/1 /content

In [3]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch.nn.functional as F
import os
from tqdm import tqdm
import numpy as np
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
import torchvision.models as models

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [5]:
train_dataset = ImageFolder(root="/content/1/Vegetable Images/train", transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [6]:
val_dataset = ImageFolder(root="/content/1/Vegetable Images/validation", transform=test_transform)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    num_workers=2,
    pin_memory=True
)

In [7]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()
        self.model = models.resnet18(weights=None)
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.model(x)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

model = CNN(num_classes=len(os.listdir("/content/vegetable-image-dataset/Vegetable Images/train")))
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

cuda


In [ ]:
epochs = 15
train_losses = []
val_losses = []

for epoch in range(epochs):
    # train
    model.train()
    train_loss = []
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
    for inputs, targets in pbar:
        optimizer.zero_grad()

        inputs, targets = inputs.to(device), targets.to(device)

        # forward
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        train_loss.append(loss.item())

        pbar.set_postfix({
            'loss': np.mean(train_loss),
            'lr': optimizer.param_groups[0]['lr']
        })

        # backward
        loss.backward()
        optimizer.step()

    train_losses.append(np.mean(train_loss))

    val_losses = []

    model.eval()
    val_loss = []
    accuracy_scores = []
    recall_scores = []
    precision_scores = []
    f1_scores = []
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss.append(loss.item())
            acc = accuracy_score(targets.cpu().numpy(), outputs.argmax(dim=-1).cpu().numpy())
            accuracy_scores.append(acc)
            recall = recall_score(targets.cpu().numpy(), outputs.argmax(dim=-1).cpu().numpy(), average='macro', zero_division=0)
            recall_scores.append(recall)
            precision = precision_score(targets.cpu().numpy(), outputs.argmax(dim=-1).cpu().numpy(), average='macro', zero_division=0)
            precision_scores.append(precision)
            f1 = f1_score(targets.cpu().numpy(), outputs.argmax(dim=-1).cpu().numpy(), average='macro', zero_division=0)
            f1_scores.append(f1)

    val_losses.append(np.mean(val_loss))

    print('Val Loss: ', np.mean(val_losses))
    print("Accuracy: ", np.mean(accuracy_scores))
    print("Recall: ", np.mean(recall_scores))
    print("Precision: ", np.mean(precision_scores))
    print("F1: ", np.mean(f1_scores))

Epoch 1/15: 100%|██████████| 469/469 [00:48<00:00,  9.63it/s, loss=1.03, lr=0.001]


Val Loss:  2.326927875247883
Accuracy:  0.4308510638297872
Recall:  0.20814296795846
Precision:  0.305551168842163
F1:  0.22427673082873398


Epoch 2/15: 100%|██████████| 469/469 [00:47<00:00,  9.93it/s, loss=0.501, lr=0.001]


Val Loss:  0.752481931149266
Accuracy:  0.7764849290780141
Recall:  0.34044241810199255
Precision:  0.3908984219090603
F1:  0.35997999683613563


Epoch 3/15: 100%|██████████| 469/469 [00:47<00:00,  9.83it/s, loss=0.327, lr=0.001]


...

Val Loss:  0.05065979097986549
Accuracy:  0.9830452127659575
Recall:  0.7958222517730497
Precision:  0.8031914893617021
F1:  0.7993915414835803


Epoch 14/15: 100%|██████████| 469/469 [00:47<00:00,  9.93it/s, loss=0.0438, lr=0.001]


Val Loss:  0.11593937727129161
Accuracy:  0.9580008865248226
Recall:  0.7338338504728132
Precision:  0.7488770685579195
F1:  0.7404007856467395


Epoch 15/15: 100%|██████████| 469/469 [00:47<00:00,  9.91it/s, loss=0.0629, lr=0.001]


Val Loss:  0.19936188463473994
Accuracy:  0.9492464539007093
Recall:  0.6979018912529551
Precision:  0.7156028368794326
F1:  0.7057186724457534


In [8]:
!git clone https://github.com/im-xiaoming/firework.git

Cloning into 'firework'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 15 (delta 3), reused 14 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), done.
Resolving deltas: 100% (3/3), done.


In [10]:
from firework.loss import CrossEntropyLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

model = CNN(num_classes=len(os.listdir("/content/1/Vegetable Images/train")))
model.to(device)

criterion = CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

cuda


In [11]:
epochs = 15
train_losses = []
val_losses = []

for epoch in range(epochs):
    # train
    model.train()
    train_loss = []
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
    for inputs, targets in pbar:
        optimizer.zero_grad()

        inputs, targets = inputs.to(device), targets.to(device)

        # forward
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        train_loss.append(loss.item())

        pbar.set_postfix({
            'loss': np.mean(train_loss),
            'lr': optimizer.param_groups[0]['lr']
        })

        # backward
        loss.backward()
        optimizer.step()

    train_losses.append(np.mean(train_loss))

    val_losses = []

    model.eval()
    val_loss = []
    accuracy_scores = []
    recall_scores = []
    precision_scores = []
    f1_scores = []
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss.append(loss.item())
            acc = accuracy_score(targets.cpu().numpy(), outputs.argmax(dim=-1).cpu().numpy())
            accuracy_scores.append(acc)
            recall = recall_score(targets.cpu().numpy(), outputs.argmax(dim=-1).cpu().numpy(), average='macro', zero_division=0)
            recall_scores.append(recall)
            precision = precision_score(targets.cpu().numpy(), outputs.argmax(dim=-1).cpu().numpy(), average='macro', zero_division=0)
            precision_scores.append(precision)
            f1 = f1_score(targets.cpu().numpy(), outputs.argmax(dim=-1).cpu().numpy(), average='macro', zero_division=0)
            f1_scores.append(f1)

    val_losses.append(np.mean(val_loss))

    print('Val Loss: ', np.mean(val_losses))
    print("Accuracy: ", np.mean(accuracy_scores))
    print("Recall: ", np.mean(recall_scores))
    print("Precision: ", np.mean(precision_scores))
    print("F1: ", np.mean(f1_scores))

Epoch 1/15: 100%|██████████| 469/469 [00:47<00:00,  9.88it/s, loss=1.02, lr=0.001]


Val Loss:  1.766196126792025
Accuracy:  0.620345744680851
Recall:  0.20021345195879767
Precision:  0.2636912697548955
F1:  0.2210560118168621


Epoch 2/15: 100%|██████████| 469/469 [00:46<00:00, 10.03it/s, loss=0.504, lr=0.001]


Val Loss:  0.7230920634168061
Accuracy:  0.7742686170212766
Recall:  0.2778023155184059
Precision:  0.33433329869500084
F1:  0.2998828466421836


Epoch 3/15: 100%|██████████| 469/469 [00:49<00:00,  9.52it/s, loss=0.349, lr=0.001]


...

Val Loss:  0.4412429944557594
Accuracy:  0.8777703900709221
Recall:  0.420034933299561
Precision:  0.4519841269841271
F1:  0.4340993831901915


Epoch 14/15: 100%|██████████| 469/469 [00:49<00:00,  9.55it/s, loss=0.059, lr=0.001]


Val Loss:  0.0297533534766386
Accuracy:  0.9913563829787234
Recall:  0.8834681589834515
Precision:  0.8874113475177304
F1:  0.8853939201070271


Epoch 15/15: 100%|██████████| 469/469 [00:49<00:00,  9.54it/s, loss=0.0664, lr=0.001]


Val Loss:  0.6465065444849353
Accuracy:  0.8424202127659575
Recall:  0.3913189167510975
Precision:  0.44181801024729694
F1:  0.41066187513269936
